# Data Quality Validation Framework

## Phase 2: Validation Layer

In this notebook, we are building a scalable enterprise-grade validation framework for the NYC Taxi dataset.

The primary objectives are:

- validate raw Bronze layer data
- identify data quality issues
- generate validation status indicators
- detect anomalous records
- prepare datasets for Silver layer processing

This validation layer forms the core of the AI-powered data quality monitoring platform.

## Step 1: Import Required Libraries

We are importing PySpark functions required for:
- validation logic
- aggregation
- anomaly detection
- quality status generation

In [0]:
from pyspark.sql import SparkSession

## Step 2: Load Raw Dataset from Bronze Layer

We are loading the raw NYC Taxi dataset from the Bronze layer volume.

The Bronze layer contains:
- raw source data
- unmodified ingestion files
- original dataset structure

In [0]:
df = spark.read.parquet("/Volumes/taxi_quality_platform/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet")

## Step 3: Display Dataset Overview

We are displaying:
- sample records
- schema structure
- dataset dimensions

This helps verify successful data loading before validations.

In [0]:
df.show(5)
df.printSchema()
df.describe()

#### Structural validation

In [0]:
# Import the required libraries

from pyspark.sql import *
from pyspark.sql import functions as f
from pyspark.sql.types import *


## Step 4: Validate Mandatory Columns

We are validating the existence of critical business columns required for downstream processing.

In [0]:
# validating the important columns exists 
df.select("VendorID", "fare_amount", "trip_distance", "payment_type", "passenger_count").show()

In [0]:
required_columns = ["VendorID", "fare_amount", "trip_distance", "payment_type", "passenger_count"]

missing_columns = [x for x in required_columns if x not in df.columns]

if len(missing_columns) > 0:
  raise Exception(f"Missing columns: {missing_columns}")



## Step 5: Validate Column Datatypes

Datatype validation helps ensure consistency and prevents downstream processing issues.

In [0]:
required_columns = ["VendorID", "fare_amount", "trip_distance", "payment_type", "passenger_count"]

for c in required_columns:
    print(type(c))

### COMPLETENESS VALIDATION

# Completeness Validations

Completeness validations help identify:
- null values
- missing fields
- incomplete records

These are common data quality issues in enterprise pipelines.

## Step 6: Null Value Validation

We are calculating null counts for important business columns.

In [0]:
from pyspark.sql.functions import col, when, count, sum

required_columns = ["VendorID", "fare_amount", "trip_distance", "payment_type", "passenger_count"]

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in required_columns
]).show()


## Step 7: Empty String Validation

Empty strings may indicate incomplete or corrupted records.

In [0]:
from pyspark.sql.functions import *

required_columns = ["VendorID", "fare_amount", "trip_distance", "payment_type", "passenger_count"]

df.select([
    sum((trim(col(c)) == "").cast("int")).alias(c)
    for c in required_columns 
]).show()

### SECTION 3: BUSINESS RULES VALIDATIONS 

# Business Rule Validations

Business validations ensure that records follow expected operational and transactional rules.

## Step 8: Fare Amount Validation

Fare amount should:
- not be null
- be greater than zero

In [0]:
df.filter((col("fare_amount") > 0) & (col("fare_amount").isNotNull())).show()

In [0]:
df.select([
    sum(((col("fare_amount") > 0) & col("fare_amount").isNotNull()).cast("int")).alias("fare_amount")
]).show()

In [0]:
# create fare validation column 

df = df.withColumn("valid_fare", ((col("fare_amount") > 0) & col("fare_amount").isNotNull()))

## Step 9: Trip Distance Validation

Trip distance should:
- not be null
- be greater than zero

In [0]:
df.select([
    sum(((col("trip_distance") > 0) & col("trip_distance").isNotNull()).cast("int")).alias("trip_distance_not_null")
]).show()

In [0]:
df = df.drop("valid")

In [0]:
# create validation flag column
df = df.withColumn("valid_distance", (col("trip_distance") > 0) & col("trip_distance").isNotNull())

## Step 10: Passenger Count Validation

Passenger count should:
- not be null
- be greater than zero

In [0]:
df.select([
    sum((((col("passenger_count").cast("int")) > 0) & col("passenger_count").cast("int").isNotNull()).cast("int")).alias("passenger_count_not_null")
]).show()

In [0]:
df = df.withColumn("valid_passenger", (((col("passenger_count").cast("int")) > 0) & col("passenger_count").cast("int").isNotNull()))

# Temporal Validations

Temporal validations ensure consistency in timestamp-based business events.

## Step 12: Pickup and Dropoff Timestamp Validation

Dropoff time should always occur after pickup time.

In [0]:
from pyspark.sql.functions import *
df.filter(col("tpep_dropoff_datetime") < col("tpep_pickup_datetime")).count()

In [0]:
# Create a validation flag for datatime 

df = df.withColumn("valid_pickup_drop", col("tpep_dropoff_datetime") < col("tpep_pickup_datetime"))

### Duplicate Validations 

# Duplicate Validations

Duplicate records can negatively impact:
- reporting accuracy
- analytics reliability
- operational monitoring

## Step 14: Duplicate Record Validation

We are identifying duplicate records within the dataset.

In [0]:
for c in df.columns:
    duplicate_count = (
        df.groupBy(c)
        .count()
        .filter(col("count") > 1).count()
    )

    print(f"{c} -> {duplicate_count} duplicate values")

In [0]:
# chheck the maximum value of fare_amount

value = df.select(max("fare_amount")).collect()[0][0]

print(value)

In [0]:
stats = df.select(
    mean("fare_amount").alias("mean"),
    stddev("fare_amount").alias("std")
).collect()[0]

print(stats)


In [0]:
# Anything beyond mean + 3*std is considered as unexpected values 
threshold = (stats[0] + (3 * stats[1]))
# print(threshold)
df.filter(col("fare_amount") > threshold).count()
# df.select(col("fare_amount")).count()

### High fare amount anomoly flag 

In [0]:
df = df.withColumn("valid_fare_amount", col("fare_amount") > threshold)

# Validation Phase Summary

In this notebook, we:
- implemented enterprise validation rules
- generated validation status indicators
- detected anomalous and invalid records
- prepared datasets for Silver layer processing

The validated data is now ready for downstream transformations and Delta table generation.